# Softmax Regression

**NOTE**: we use row vector convention

Predicts the probability for a given k-class.

**Inputs**
- $X \in [B, D]$
- $Y \in [B, K]$

**Model**

- $W \in [D, K]$

- $b \in [1, K]$

$$z = X W + b \in [B, K]$$

$$P(z) = softmax(z) \in [B, K] $$

For each class $j \in [1, K]$, 
$$p(z_j) = \frac{e^{z_j}}{\sum_{k=1}^K e^{z_k}}$$

**Loss**

Batched cross entropy

$$
\begin{aligned}
\mathbb{L} &= -\frac{1}{B} \sum_i^B \sum_j^K y_{i, j} \log (p(z_{i, j})) \\
& = \frac{1}{B} \sum_i^B l_i \rightarrow l_i \text{ is loss for sample } i \\
& = -\frac{1}{B} \sum_i^B y_{i}^T \log (p(z_{i}))
\end{aligned}
$$



## Gradient Derivation ##

$$
\frac{d\mathbb{L}}{dW} = \frac{\partial \mathbb{L}}{\partial z}\frac{\partial z}{\partial W}
$$

$$
\frac{d\mathbb{L}}{db} = \frac{\partial \mathbb{L}}{\partial z}\frac{\partial z}{\partial b}
$$



Use the following derivatives:

### Derivative of loss function ###
From row-wise (per-smaple) perspective, the batched gradients of loss w.r.t. $z$ is:

$$ 
\begin{aligned}
\frac{\partial \mathbb{L}}{\partial z} = \frac{1}{B}\begin{bmatrix}
\frac{\partial l_1}{\partial z_1} \\
\frac{\partial l_2}{\partial z_2} \\
\vdots \\
\frac{\partial l_B}{\partial z_B} \\
\end{bmatrix}

\end{aligned}
$$

The loss per sample $i$ is:

$$
\begin{aligned}
\frac{\partial l_i}{\partial z_i} & = \frac{\partial l_i}{\partial p(z_i)} \frac{\partial p(z_i)}{\partial z_i}
\end{aligned}
$$

For simplicity, we remove the denotion of $i$, the per-sample loss gradient w.r.t. $z$ is then: 
$$\frac{\partial l}{\partial z} = \frac{\partial l}{\partial p(z)} \frac{\partial p(z)}{\partial z}$$

where $z, p \in \mathbb{R}^{1 \times K}$

### Derivative of cross entropy loss w.r.t. $p(z)$ ###

$$
\begin{aligned}
\frac{\partial l}{\partial p(z)} & = \frac{\partial}{\partial p(z)} \left( -y^T \log (p(z)) \right) \\
& = -\frac{y}{p(z)}
\end{aligned}
$$



### Derivatives of softmax function ###
Softmax function $p(z)$ is a vector function, which maps a vector $z$ to another vector $p(z)$. The Jacobian matrix of softmax function $\frac{\partial p}{\partial z}$ is a [K, K] matrix for each sample:

$$
\frac{\partial p}{\partial z} = 
\begin{Bmatrix}
\frac{\partial p_{0}}{\partial z_{0}}, &\frac{\partial p_{0}}{\partial z_{1}}, &..., &\frac{\partial p_{0}}{\partial z_{k-1}} \\
\frac{\partial p_{1}}{\partial z_{0}}, &\frac{\partial p_{1}}{\partial z_{1}}, &..., &\frac{\partial p_{1}}{\partial z_{k-1}} \\
..., &..., &..., &... \\
\frac{\partial p_{k-1}}{\partial z_{0}}, &\frac{\partial p_{k-1}}{\partial z_{1}}, &..., &\frac{\partial p_{k-1}}{\partial z_{k-1}} \\
\end{Bmatrix}_{K \times K}
$$

For given element $\frac{\partial p_i}{\partial z_j}$:
- Case 1: $i == j$

    $$
    \begin{aligned}
    \frac{\partial p_i}{\partial z_i} &= \frac{\partial \frac{e^{z_i}}{\sum_1^k e^z_h}}{\partial z_i} \\
        &=\frac{e^{z_i}}{\sum_1^k e^{z_h}} - (\frac{e^{z_i}}{\sum_1^k e^{z_h}})^2 \\
        &= p_i - p_i^2 \\
        &= p_i(1-p_i) \\
    \end{aligned}
    $$

- Case 2: $i \neq j$

    $$
    \begin{aligned}
    \frac{\partial p_i}{\partial z_j} &= \frac{\partial \frac{e^{z_i}}{\sum_1^k e^z_h}}{\partial z_j} \\
        &=0 - \frac{e^{z_i}e^{z_j}}{(\sum_1^k e^{z_h})^2} \\
        &= -p_ip_j\\
    \end{aligned}
    $$

Combing these two cases together:

$$
\frac{\partial p_i}{\partial z_j} = p_i(\delta_{i,j} - p_j)
$$
where $\delta_{ij} = 1$ if $i == j$ else 0.


To write the above in a vectorized form, the Jacobian matrix of softmax function can be written as:
$$
\begin{aligned}
\frac{\partial p}{\partial z} & = J = diag(p) - p^T p
\end{aligned}
$$



### Derivative of loss w.r.t. logits $z$ ###

$$
\begin{aligned}
\frac{\partial l}{\partial z} & = \frac{\partial l}{\partial p(z)} \frac{\partial p(z)}{\partial z} \\
& = -\frac{y}{p(z)} (diag(p) - p^T p) \\
& = -\frac{y}{p} diag(p) + \frac{y}{p} p^T p \\
& = -y + (\sum_i \frac{y_i}{p_i} p_i) p \\
& = -y + (\sum_i y_i) p \rightarrow \text{y is one-hot encoded}\\
& = -y + p \\ 
\end{aligned}
$$


### Derivatives of Linear Function ###
$$
\frac{\partial z}{\partial W} = X
$$

$$
\frac{\partial z}{\partial b} = 1|_{(B \times 1)}
$$


### Final Gradients ###
Combing the softmax gradient and the cross-entropy loss gradient together help reduce the needs of calculating the Jacobian matrix of softmax function. 


$$ 
\begin{aligned}
\frac{\partial \mathbb{L}}{\partial z} = \frac{1}{B}\begin{bmatrix}
\frac{\partial l_1}{\partial z_1} \\
\frac{\partial l_2}{\partial z_2} \\
\vdots \\
\frac{\partial l_B}{\partial z_B} \\
\end{bmatrix}
= \frac{1}{B}\begin{bmatrix}
p_1 - y_1 \\
p_2 - y_2 \\
\vdots \\
p_B - y_B \\
\end{bmatrix}_{B \times K}
= \frac{1}{B} (P - Y)
\end{aligned}
$$


the final gradients of loss w.r.t. $W$ and $b$ can be written as:

$$
\begin{aligned}
\frac{d\mathbb{L}}{dW} & = (\frac{\partial z}{\partial W})^T \frac{\partial \mathbb{L}}{\partial z} \\
& = \frac{1}{B} X^T (P - Y) \\
\frac{d\mathbb{L}}{db} & = (\frac{\partial z}{\partial b})^T \frac{\partial \mathbb{L}}{\partial z} \\
& = \frac{1}{B} 1^T (P - Y) \\
& = \frac{1}{B} \sum_i (p_i - y_i)
\end{aligned}
$$




## Data and Settings

In [15]:
import torch 
import numpy as np

In [16]:
B, D, K = 100, 10, 3

X = np.random.randn(B, D)
Y = np.random.randn(B, K)

# generate multinomial labels
Y = np.exp(Y)
Y = Y / np.sum(Y, axis=1, keepdims=True)
Y = np.argmax(Y, axis=1)

print(Y)

max_iters = 1000
lr = 1e-2

[1 2 2 0 1 1 2 2 1 0 0 1 1 2 0 1 1 0 0 0 2 1 0 2 2 1 0 0 0 2 0 0 0 0 0 0 0
 2 0 0 2 2 0 1 2 0 0 2 1 0 0 2 1 2 0 0 1 1 0 1 2 0 2 0 0 2 2 0 1 0 1 1 1 2
 0 0 1 2 2 2 2 1 2 0 2 0 0 1 0 0 0 0 2 2 1 0 0 0 2 2]


## Numpy Implementation

In [17]:
def one_hot(y, K):
    """
    Args:
        y: (B,)
        K: scalar
    Returns:
        (B, K)
    """
    B = y.shape[0]
    one_hot = np.zeros((B, K))
    one_hot[np.arange(B), y] = 1
    
    return one_hot


def softmax(z):
    """
    Args:
        z: (B, K)
    Returns:
        (B, K)
    """
    z_max = np.max(z, axis=1, keepdims=True)
    exp_z = np.exp(z - z_max)
    p = exp_z / np.sum(exp_z, axis=1, keepdims=True)
    
    return p

def model(x, W, b):
    """
    Args:
        x: (B, D)
        W: (D, K)
        b: (1, K)
    Returns:
        (B, K)
    """
    z = x @ W + b
    
    return z

def cross_entropy_with_softmax(logits, target):
    """
    Args:
        logits: (B, K)
        target: (B, K)
    Returns:
        scalar
    """
    eps = 1e-8
    pred = softmax(logits)
    loss = - np.sum(target * np.log(pred + eps), axis=1)
    
    return np.mean(loss)

def gradients(pred, target, x):
    """
    Args:
        pred: (B, K)
        target: (B, K)
        x: (B, D) 
    """
    B, _= pred.shape
    
    dw = 1/B * x.T @ (pred - target)
    db = 1/B * np.sum(pred - target, axis=0, keepdims=True)
    
    
    return dw, db


In [18]:
# TRAINING 
# initialize weights
W = np.random.randn(D, K)
b = np.random.randn(1, K)

# one-hot encode labels
y = one_hot(Y, K)

for i in range(max_iters):
    # forward 
    logits = model(X, W, b)
    pred = softmax(logits)
    
    # loss 
    loss = cross_entropy_with_softmax(logits, y)

    # backward for gradient computation
    dw, db = gradients(pred, y, X)

    # update weights
    W -= lr * dw
    b -= lr * db

    # log 
    if i % 100 == 0:
        print(f"iter {i}: loss {loss:.4f}")

iter 0: loss 2.4205
iter 100: loss 2.0891
iter 200: loss 1.8299
iter 300: loss 1.6317
iter 400: loss 1.4815
iter 500: loss 1.3672
iter 600: loss 1.2792
iter 700: loss 1.2100
iter 800: loss 1.1551
iter 900: loss 1.1112


## Pytorch Implementation


In [19]:
class LinearRegression(torch.nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.linear = torch.nn.Linear(in_features, out_features)
    
    def forward(self, x):
        return self.linear(x)


max_iters = 1000
lr = 1e-2

loss_fcn = torch.nn.CrossEntropyLoss()
model = LinearRegression(D, K)
optimizer = torch.optim.SGD(model.parameters(), lr=lr)

In [20]:
XTensor = torch.from_numpy(X).float()
YTensor = torch.from_numpy(Y).long()

for i in range(max_iters):
    # forward 
    logits = model(XTensor)
    
    # loss 
    loss = loss_fcn(logits, YTensor)
    
    # backward for gradient computation
    loss.backward()
    
    # update weights
    optimizer.step()
    
    # zero gradients
    optimizer.zero_grad()
    
    # log 
    if i % 100 == 0:
        print(f"iter {i}: loss {loss.item():.4f}")

iter 0: loss 1.2015
iter 100: loss 1.0873
iter 200: loss 1.0299
iter 300: loss 0.9999
iter 400: loss 0.9833
iter 500: loss 0.9734
iter 600: loss 0.9672
iter 700: loss 0.9632
iter 800: loss 0.9605
iter 900: loss 0.9586
